# **R/S BENCHMARK — RUL DATASET GENERATION**

## Prerequisite

Run [`01_generate_dataset.ipynb`](01_generate_dataset.ipynb) $\to$
[`02_train_pce.ipynb`](02_train_pce.ipynb) $\to$
[`03_generate_dataset_nn.ipynb`](03_generate_dataset_nn.ipynb) $\to$
[`04_train_nn.ipynb`](04_train_nn.ipynb) first — this notebook loads the global NN
(`lambda 1`/`lambda 2` vs. $(R, S, t)$) that stage 4 writes.

## What this notebook does

Fixes a single design point $(R, S)$ and sweeps a list of time steps through the trained NN, giving
$\lambda_1(t)$ and $\lambda_2(t)$ at each one **directly** — no need to pick a per-time-step PCE
first. $\lambda_3$ and $\lambda_4$ barely move across the design space (see the $\lambda$ maps in
[`01_plot_lambda_maps.ipynb`](01_plot_lambda_maps.ipynb)), which is why they were never modelled by
the NN — they're fixed here instead, to a value you set or, by default, the mean over the NN's own
training dataset.

At each time step, `generate_rul_dataset_benchmark` builds a
`GlamFKML(lam1, lam2, lam3, lam4)` and draws `n_glam_samples` Monte Carlo realisations of the state
limit function $g$ — the raw material for the spaghetti plot and the RUL analysis in
[`06_plot_rul_analysis.ipynb`](06_plot_rul_analysis.ipynb).

## 1. Libraries

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import dill
import numpy as np
import pandas as pd

from functions import *

/home/casa-wand/steam2tb/2024-1_victor_hugo_renata_maria/.venv/lib/python3.11/site-packages/UQpy/__init__.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


## 2. Config

`n_latent_samples` must match [`04_train_nn.ipynb`](04_train_nn.ipynb) — it names the NN model
files being loaded.

`lambda3_fixed`/`lambda4_fixed` left as `None` fall back to the mean of `lambda 3`/`lambda 4` over
the NN training dataset; set either explicitly to override.

In [2]:
n_latent_samples = 2500   # must match stage 4 — it names the NN model files being loaded

r_fixed = 5.0   # fixed R for the whole sweep
s_fixed = 2.0   # fixed S for the whole sweep
times   = np.linspace(0, 100, 20, endpoint=True)  # time steps to sweep through the NN — any grid, not just the training one

lambda3_fixed = 0.142157   # None = mean of lambda 3 over the NN training dataset
lambda4_fixed = 0.131525   # None = mean of lambda 4 over the NN training dataset

n_glam_samples = 50000   # Monte Carlo samples drawn from the GLD at each time step

print(f"Sweeping t = {times.min()}..{times.max()} years at R={r_fixed}, S={s_fixed}")

Sweeping t = 0.0..100.0 years at R=5.0, S=2.0


## 3. Predict lambda 1/2, fix lambda 3/4, and draw the GLD samples

In [3]:
print("="*60)
print("GENERATING THE RUL DATASET")
print("="*60)

result = generate_rul_dataset_benchmark(
                                           r=r_fixed,
                                           s=s_fixed,
                                           times=times,
                                           n_latent_samples=n_latent_samples,
                                           lambda3_fixed=lambda3_fixed,
                                           lambda4_fixed=lambda4_fixed,
                                           n_glam_samples=n_glam_samples,
                                           input_dir='.',
                                           output_dir='.',
                                        )

lambda_df = result['lambda_df']
samples   = result['samples']
print(f"\nSamples shape: {samples.shape}  (n_glam_samples x len(times))")
lambda_df

GENERATING THE RUL DATASET

----------------------------------------
GENERATING RUL DATASET AT R=5.0, S=2.0
----------------------------------------
  lambda 3 fixed at 0.1422, lambda 4 fixed at 0.1315
  t = 0.0: lambda 1 = 2.996, lambda 2 = 6.128, sample mean = 2.998, sample std = 0.238
  t = 5.3: lambda 1 = 2.866, lambda 2 = 6.176, sample mean = 2.868, sample std = 0.237
  t = 10.5: lambda 1 = 2.739, lambda 2 = 6.199, sample mean = 2.740, sample std = 0.236
  t = 15.8: lambda 1 = 2.608, lambda 2 = 6.260, sample mean = 2.610, sample std = 0.233
  t = 21.1: lambda 1 = 2.468, lambda 2 = 6.360, sample mean = 2.470, sample std = 0.230
  t = 26.3: lambda 1 = 2.345, lambda 2 = 6.427, sample mean = 2.347, sample std = 0.227
  t = 31.6: lambda 1 = 2.219, lambda 2 = 6.453, sample mean = 2.220, sample std = 0.226
  t = 36.8: lambda 1 = 2.082, lambda 2 = 6.483, sample mean = 2.084, sample std = 0.225
  t = 42.1: lambda 1 = 1.940, lambda 2 = 6.559, sample mean = 1.942, sample std = 0.223
  t = 47

,r,s,Time (years),lambda 1,lambda 2,lambda 3,lambda 4
0,5.0,2.0,0.000000,2.995682,6.128036,0.142157,0.131525
1,5.0,2.0,5.263158,2.866417,6.175774,0.142157,0.131525
2,5.0,2.0,10.526316,2.738662,6.199467,0.142157,0.131525
3,5.0,2.0,15.789474,2.608061,6.260498,0.142157,0.131525
4,5.0,2.0,21.052632,2.468088,6.360435,0.142157,0.131525
5,5.0,2.0,26.315789,2.344785,6.426558,0.142157,0.131525
6,5.0,2.0,31.578947,2.218716,6.452731,0.142157,0.131525
7,5.0,2.0,36.842105,2.082385,6.482594,0.142157,0.131525
8,5.0,2.0,42.105263,1.940453,6.558748,0.142157,0.131525
9,5.0,2.0,47.368421,1.813681,6.642764,0.142157,0.131525


## 4. Sanity check

In [4]:
summary = pd.DataFrame({
                           'Time (years)': result['times'],
                           'Sample mean':  samples.mean(axis=0),
                           'Sample std':   samples.std(axis=0),
                           'P(g <= 0)':    (samples <= 0).mean(axis=0),
                        })
summary

,Time (years),Sample mean,Sample std,P(g <= 0)
0,0.000000,2.997511,0.238402,0.00000
1,5.263158,2.868232,0.236559,0.00000
2,10.526316,2.740470,0.235655,0.00000
3,15.789474,2.609851,0.233358,0.00000
4,21.052632,2.469850,0.229691,0.00000
5,26.315789,2.346529,0.227328,0.00000
6,31.578947,2.220452,0.226406,0.00000
7,36.842105,2.084114,0.225363,0.00000
8,42.105263,1.942161,0.222746,0.00000
9,47.368421,1.815368,0.219929,0.00000


In [5]:
summary = pd.DataFrame({
                           'Time (years)': result['times'],
                           'Sample mean':  samples.mean(axis=0),
                           'Sample std':   samples.std(axis=0),
                           'P(g <= 0.2)':    (samples <= 0.2).mean(axis=0),
                        })
summary

,Time (years),Sample mean,Sample std,P(g <= 0.2)
0,0.000000,2.997511,0.238402,0.00000
1,5.263158,2.868232,0.236559,0.00000
2,10.526316,2.740470,0.235655,0.00000
3,15.789474,2.609851,0.233358,0.00000
4,21.052632,2.469850,0.229691,0.00000
5,26.315789,2.346529,0.227328,0.00000
6,31.578947,2.220452,0.226406,0.00000
7,36.842105,2.084114,0.225363,0.00000
8,42.105263,1.942161,0.222746,0.00000
9,47.368421,1.815368,0.219929,0.00000
